# conv-output-shape composite — cx15: 1D conv with stride+padding: predict shape, then build window view

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `conv-output-shape`, `conv-windowing-1d`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "conv-output-shape"
DD_ATOM_IDS = ["conv-output-shape", "conv-windowing-1d"]
DD_SUBTOPICS = ["CNN: Conv output shape", "CNN: 1-D conv windowing"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Stride-1 / no-pad conv1d lets you assume `OW = W - KW + 1`. Real ARENA conv supports **stride > 1** and **padding > 0**, so the size of the strided view depends on the output shape formula:

```
OW = (W + 2*P - KW) // S + 1
```

This is `conv-output-shape` (the 1-D specialization). Once you know `OW`, `conv-windowing-1d` builds the view — but with one twist: the OW-axis stride is now `s_w * S`, not just `s_w` (you step `S` input elements per output position).

**Pad first, then window.** PyTorch zero-padding adds a contiguous halo around `x`. After padding, the padded tensor's spatial stride is its own `s_w`; use THAT stride in `as_strided`. Padding is `F.pad(x, (P, P))` for 1-D — pads only the last axis with `P` zeros on each side.

**Anatomy of the composite.**
1. Compute `OW` from the formula. (Atom A — `conv-output-shape`.)
2. Pad `x` if `P > 0`. (Implementation detail — not an atom.)
3. Build the strided view of shape `(B, IC, OW, KW)` with `OW`-stride `= s_w * S`. (Atom B — `conv-windowing-1d`.)
4. Einsum-contract against the kernel.

### Composite Exercise — 1D conv with stride+padding: predict shape, then build window view

**Atoms exercised together**: `conv-output-shape`, `conv-windowing-1d`

Implement `cx15_strided_conv1d(x, weight, stride=1, padding=0)`.

- `x`: float tensor `(B, IC, W)`.
- `weight`: float tensor `(OC, IC, KW)`.
- `stride`, `padding`: ints.
- Return: tensor `(B, OC, OW)` matching `F.conv1d(x, weight, stride=stride, padding=padding)`.

Also implement `cx15_predict_outshape(input_shape, OC, KW, stride, padding)` returning the predicted `(B, OC, OW)` tuple — the test verifies this matches the actual `F.conv1d` output.

**Tip.** Use `F.pad(x, (padding, padding))` for the padding step. Compute `OW` ANALYTICALLY before windowing — do not derive it from `x.shape` after padding (you'd repeat the formula anyway).

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx15_predict_outshape(input_shape, OC, KW, stride, padding):
    raise NotImplementedError

def cx15_strided_conv1d(x, weight, stride=1, padding=0):
    raise NotImplementedError

def _test_cx15():
    from torch.nn import functional as F
    rng = t.Generator().manual_seed(15)

    # Case A: output-shape predictor cross-check against F.conv1d.
    for B,IC,W,OC,KW,S,P in [(1,1,10,1,3,1,0),(2,3,16,4,5,2,1),(1,1,8,1,3,1,1),(3,2,20,5,7,3,2),(1,1,7,1,1,1,0)]:
        pred = cx15_predict_outshape((B,IC,W), OC, KW, S, P)
        actual = tuple(F.conv1d(t.zeros(B,IC,W), t.zeros(OC,IC,KW), stride=S, padding=P).shape)
        assert tuple(pred) == actual, f'shape pred {pred} vs actual {actual} for {(B,IC,W,OC,KW,S,P)}'

    # Case B: full conv1d values on the same shapes.
    for B,IC,W,OC,KW,S,P in [(1,1,10,1,3,1,0),(2,3,16,4,5,2,1),(1,1,9,1,3,1,1),(3,2,21,5,7,3,2)]:
        x = t.randn(B,IC,W, generator=rng)
        w = t.randn(OC,IC,KW, generator=rng)
        yref = F.conv1d(x, w, stride=S, padding=P)
        yours = cx15_strided_conv1d(x, w, stride=S, padding=P)
        assert tuple(yours.shape) == tuple(yref.shape), f'shape mismatch on {(B,IC,W,OC,KW,S,P)}'
        assert t.allclose(yours, yref, atol=1e-4), f'value mismatch on {(B,IC,W,OC,KW,S,P)}'

    # Case C: stride==kernel, no pad → non-overlapping windows.
    x = t.randn(2, 3, 12, generator=rng)
    w = t.randn(4, 3, 4, generator=rng)
    yours = cx15_strided_conv1d(x, w, stride=4, padding=0)
    yref = F.conv1d(x, w, stride=4, padding=0)
    assert tuple(yours.shape) == (2, 4, 3)
    assert t.allclose(yours, yref, atol=1e-4)
    _dd_passed.add('cx15')

_test_cx15()

<details><summary>Show solution — cx15</summary>

```python
def cx15_predict_outshape(input_shape, OC, KW, stride, padding):
    # Atom A (conv-output-shape, 1-D form).
    B, IC, W = input_shape
    OW = (W + 2 * padding - KW) // stride + 1
    return (B, OC, OW)

def cx15_strided_conv1d(x, weight, stride=1, padding=0):
    from torch.nn import functional as F
    B, IC, W = x.shape
    OC, IC2, KW = weight.shape
    assert IC == IC2
    # Use atom A to know OW up front.
    _, _, OW = cx15_predict_outshape(x.shape, OC, KW, stride, padding)
    # Pad, then window.
    xp = F.pad(x, (padding, padding)) if padding > 0 else x
    s_b, s_ic, s_w = xp.stride()
    # Atom B (conv-windowing-1d) with stride-S step on the OW axis.
    x_win = xp.as_strided(
        size=(B, IC, OW, KW),
        stride=(s_b, s_ic, s_w * stride, s_w),
    )
    return einops.einsum(x_win, weight, 'b ic ow kw, oc ic kw -> b oc ow')
```

Two atoms, one composition: the shape formula tells you `OW`, the windowing trick builds the view sized to that `OW`. Decoupling 'what shape do I need' from 'how do I build it' is what lets you handle stride+padding without re-deriving from indices.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx15'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx15',
        'subtopics': ["CNN: Conv output shape", "CNN: 1-D conv windowing"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()